In [ ]:
# ══════════════════════════════════════════════════════════════════
# Colab setup — self-contained. No git clone, no FORK_URL, no sys.path.
# Before running: Runtime → Change runtime type → T4 GPU (free tier)
# ══════════════════════════════════════════════════════════════════
!pip install -q kailash polars plotly gdown python-dotenv nest-asyncio kailash-ml

import nest_asyncio; nest_asyncio.apply()

import torch
if torch.cuda.is_available():
    print(f'✓ GPU available: {torch.cuda.get_device_name(0)}')
else:
    print('⚠ No GPU — training will be slow. Runtime → Change runtime type → T4 GPU')

print('✓ Setup complete. All helpers defined in the next cell.')


In [ ]:
# ══════════════════════════════════════════════════════════════════
# MLFP inlined helpers — DO NOT EDIT (collapse this cell!)
# Auto-generated by scripts/generate_selfcontained_notebook.py for mlfp04
# ══════════════════════════════════════════════════════════════════
from __future__ import annotations

# ── shared/kailash_helpers.py ──
"""Common Kailash SDK setup patterns for MLFP exercises."""


import os
from pathlib import Path

from dotenv import load_dotenv


def setup_environment() -> None:
    """Load .env and validate common configuration.

    Call this at the top of every exercise that needs API keys or DB connections.
    """
    # Find .env by walking up from the exercise file
    env_path = Path.cwd() / ".env"
    if not env_path.exists():
        # Try repo root
        for parent in Path.cwd().parents:
            candidate = parent / ".env"
            if candidate.exists():
                env_path = candidate
                break

    load_dotenv(env_path)


def get_connection_manager(db_url: str | None = None):
    """Create a ConnectionManager for kailash-ml engines.

    Args:
        db_url: Database URL. Defaults to SQLite at ./mlfp.db
    """
    from kailash.db import ConnectionManager

    url = db_url or os.environ.get("DATABASE_URL", "sqlite:///mlfp.db")
    return ConnectionManager(url)


def get_device() -> "torch.device":
    """Get the best available compute device: MPS (Mac) > CUDA > CPU."""
    import torch

    if hasattr(torch.backends, "mps") and torch.backends.mps.is_available():
        return torch.device("mps")
    if torch.cuda.is_available():
        return torch.device("cuda")
    return torch.device("cpu")


def get_llm_model() -> str:
    """Get the configured LLM model name from environment."""
    setup_environment()
    model = os.environ.get("DEFAULT_LLM_MODEL", os.environ.get("OPENAI_PROD_MODEL"))
    if not model:
        raise EnvironmentError(
            "No LLM model configured. Set DEFAULT_LLM_MODEL or OPENAI_PROD_MODEL in .env"
        )
    return model


# ── shared/mlfp04/ex_1.py ──
"""
Shared infrastructure for MLFP04 Exercise 1 — Clustering Zoo.

Contains: customer-feature loading & standardisation, metric helpers,
subsampling utilities, and output-directory management.

Technique-specific code (K-means elbow, linkage methods, DBSCAN epsilon
search, HDBSCAN, spectral Laplacian, AutoMLEngine) does NOT belong
here — it lives in the per-technique files under modules/mlfp04/solutions/ex_1/.
"""

from pathlib import Path
from typing import Any

import numpy as np
import polars as pl
from sklearn.metrics import (
    adjusted_rand_score,
    calinski_harabasz_score,
    davies_bouldin_score,
    normalized_mutual_info_score,
    silhouette_score,
)
from sklearn.preprocessing import StandardScaler

from kailash_ml.interop import to_sklearn_input


# ════════════════════════════════════════════════════════════════════════
# ENVIRONMENT SETUP
# ════════════════════════════════════════════════════════════════════════

setup_environment()

OUTPUT_DIR = Path("outputs") / "ex1_clustering"
OUTPUT_DIR.mkdir(parents=True, exist_ok=True)

# Shared random state so every technique file is reproducible
RANDOM_STATE: int = 42


# ════════════════════════════════════════════════════════════════════════
# DATA LOADING — Singapore e-commerce customers
# ════════════════════════════════════════════════════════════════════════


def load_customers() -> tuple[pl.DataFrame, list[str]]:
    """Load the e-commerce customer dataset and return (df, numeric_feature_cols).

    The dataset (from MLFP03) is ~6K rows of Singapore e-commerce customers
    with recency, frequency, monetary, basket-size, and channel features.
    Clustering is unsupervised segmentation: no labels, just behaviour.
    """
    loader = MLFPDataLoader()
    customers = loader.load("mlfp03", "ecommerce_customers.parquet")
    numeric_types = (pl.Float64, pl.Float32, pl.Int64, pl.Int32)
    feature_cols = [
        c
        for c, d in zip(customers.columns, customers.dtypes)
        if d in numeric_types and c not in ("customer_id",)
    ]
    return customers.drop_nulls(subset=feature_cols), feature_cols


def standardise(
    df: pl.DataFrame, feature_cols: list[str]
) -> tuple[np.ndarray, StandardScaler]:
    """Return (X_scaled, scaler). Zero mean, unit variance — mandatory for
    all distance-based clustering."""
    X, _, _ = to_sklearn_input(df, feature_columns=feature_cols)
    scaler = StandardScaler()
    X_scaled = scaler.fit_transform(X)
    return X_scaled, scaler


# ════════════════════════════════════════════════════════════════════════
# SUBSAMPLING — spectral / hierarchical are O(n^2) or worse
# ════════════════════════════════════════════════════════════════════════


def subsample(
    X: np.ndarray, n: int, seed: int = RANDOM_STATE
) -> tuple[np.ndarray, np.ndarray]:
    """Return (X_sub, idx) where idx are the original row indices chosen."""
    rng = np.random.default_rng(seed)
    n = min(n, X.shape[0])
    idx = rng.choice(X.shape[0], n, replace=False)
    return X[idx], idx


# ════════════════════════════════════════════════════════════════════════
# METRICS
# ════════════════════════════════════════════════════════════════════════


def score_partition(X: np.ndarray, labels: np.ndarray) -> dict[str, float]:
    """Compute silhouette, Calinski-Harabasz, Davies-Bouldin.

    Points with label == -1 (DBSCAN noise) are excluded. Returns NaN
    fields if fewer than 2 valid clusters remain.
    """
    valid = labels != -1
    labs = labels[valid]
    data = X[valid]
    if data.shape[0] < 2 or len(set(labs.tolist())) < 2:
        return {
            "n_clusters": len(set(labs.tolist())),
            "silhouette": float("nan"),
            "calinski_harabasz": float("nan"),
            "davies_bouldin": float("nan"),
        }
    return {
        "n_clusters": len(set(labs.tolist())),
        "silhouette": float(silhouette_score(data, labs)),
        "calinski_harabasz": float(calinski_harabasz_score(data, labs)),
        "davies_bouldin": float(davies_bouldin_score(data, labs)),
    }


def agreement(labels_a: np.ndarray, labels_b: np.ndarray) -> dict[str, float]:
    """ARI and NMI between two label vectors, ignoring noise (-1)."""
    valid = (labels_a >= 0) & (labels_b >= 0)
    if valid.sum() < 2:
        return {"ari": float("nan"), "nmi": float("nan")}
    return {
        "ari": float(adjusted_rand_score(labels_a[valid], labels_b[valid])),
        "nmi": float(normalized_mutual_info_score(labels_a[valid], labels_b[valid])),
    }


def print_metric_row(name: str, m: dict[str, Any]) -> None:
    """One-line summary of a partition's metrics."""
    print(
        f"  {name:<14} K={m['n_clusters']:>3}  "
        f"sil={m['silhouette']:>7.4f}  "
        f"CH={m['calinski_harabasz']:>9.0f}  "
        f"DB={m['davies_bouldin']:>7.4f}"
    )


# ════════════════════════════════════════════════════════════════════════
# VISUALISATION OUTPUT PATH
# ════════════════════════════════════════════════════════════════════════


def out_path(filename: str) -> Path:
    """Return a path under OUTPUT_DIR for a visualisation artefact."""
    return OUTPUT_DIR / filename




# ════════════════════════════════════════════════════════════════════════
# MLFP04 — Exercise 1.1: K-means with k-means++ Initialisation
# ════════════════════════════════════════════════════════════════════════
#
# WHAT YOU'LL LEARN:
#   - Apply K-means with k-means++ initialisation and understand why it
#     converges faster than random initialisation
#   - Use the elbow method and silhouette score to select K objectively
#   - Read per-sample silhouette to spot mis-assigned points
#   - Interpret inertia (within-cluster sum of squares) as a loss value
#
# PREREQUISITES: MLFP03 complete (supervised ML, feature scaling).
#
# ESTIMATED TIME: ~30 min
#
# TASKS:
#   1. Theory — why K-means works and how k-means++ fixes its weakness
#   2. Build — the elbow + silhouette sweep across K
#   3. Train — fit K-means with k-means++ vs random and compare
#   4. Visualise — silhouette curves vs K + per-sample silhouette
#   5. Apply — Singapore Shopee loyalty segmentation, $ impact per tier
# ════════════════════════════════════════════════════════════════════════


In [ ]:
from __future__ import annotations

import time

import numpy as np
from dotenv import load_dotenv
from sklearn.cluster import KMeans
from sklearn.metrics import (
    calinski_harabasz_score,
    davies_bouldin_score,
    silhouette_samples,
    silhouette_score,
)

from kailash_ml import ModelVisualizer


load_dotenv()



## THEORY — Why K-means Works and Why k-means++ Matters


In [ ]:
# K-means minimises the within-cluster sum of squares:
#     J = Σ_k Σ_{x in C_k} ||x - μ_k||²
# It alternates two steps: assign each point to the nearest centroid, then
# recompute each centroid as the mean of its assigned points. This is a
# coordinate-descent algorithm — each step can only decrease J — so it
# converges in a finite number of iterations.
#
# The catch: J is non-convex. Different starting centroids converge to
# different local minima. Random initialisation occasionally places two
# centroids on top of each other and ends up with an empty cluster or a
# very poor partition.
#
# k-means++ fixes this by spreading the initial centroids apart:
#   1. Pick the first centroid uniformly at random.
#   2. For each subsequent centroid, sample a point with probability
#      proportional to its squared distance from the nearest existing
#      centroid.
# The result is a provably O(log K) approximation to the optimal seeding,
# and in practice it converges 2-5× faster and to a lower final J.



## TASK 2 — BUILD: Load data and sweep K with silhouette scoring


Fit K-means for each K and return per-K inertia and validity metrics.


In [ ]:
customers, feature_cols = load_customers()
X_scaled, _ = standardise(customers, feature_cols)
n_samples, n_features = X_scaled.shape

print("=" * 70)
print("  K-means on Singapore E-commerce Customers")
print("=" * 70)
print(f"  Samples={n_samples:,}  features={n_features}")
print(f"  Feature columns: {feature_cols}")


def sweep_k(X: np.ndarray, k_values: range) -> dict[str, list[float]]:
    inertias, sils, chs, dbs = [], [], [], []
    print(f"\n  {'K':>3} {'Inertia':>12} {'Silhouette':>12} {'CH':>10} {'DB':>8}")
    print("  " + "─" * 50)
    for k in k_values:
        km = KMeans(
            n_clusters=k, random_state=RANDOM_STATE, n_init=10, init="k-means++"
        )
        labels = km.fit_predict(X)
        inertias.append(km.inertia_)
        sils.append(silhouette_score(X, labels))
        chs.append(calinski_harabasz_score(X, labels))
        dbs.append(davies_bouldin_score(X, labels))
        print(
            f"  {k:>3} {km.inertia_:>12.0f} {sils[-1]:>12.4f} "
            f"{chs[-1]:>10.0f} {dbs[-1]:>8.4f}"
        )
    return {"inertia": inertias, "silhouette": sils, "ch": chs, "db": dbs}


K_RANGE = range(2, 11)
sweep = sweep_k(X_scaled, K_RANGE)
best_k = list(K_RANGE)[int(np.argmax(sweep["silhouette"]))]
print(f"\n  Best K by silhouette: {best_k} (score={max(sweep['silhouette']):.4f})")



### Checkpoint 1


In [ ]:
assert 2 <= best_k <= 10, "Task 2: best_k must be in the tested range"
assert max(sweep["silhouette"]) > 0, "Task 2: best silhouette should be positive"
assert len(sweep["inertia"]) == len(list(K_RANGE)), "Task 2: sweep size mismatch"
print("\n  [ok] Checkpoint 1 passed — silhouette sweep complete\n")



## TASK 3 — TRAIN: k-means++ vs random initialisation head-to-head


In [ ]:
km_plus = KMeans(
    n_clusters=best_k, random_state=RANDOM_STATE, n_init=10, init="k-means++"
)
km_random = KMeans(
    n_clusters=best_k, random_state=RANDOM_STATE, n_init=10, init="random"
)

t0 = time.perf_counter()
km_plus.fit(X_scaled)
t_plus = time.perf_counter() - t0

t0 = time.perf_counter()
km_random.fit(X_scaled)
t_random = time.perf_counter() - t0

print(f"  k-means++ vs Random Initialisation (K={best_k}):")
print(
    f"    k-means++: inertia={km_plus.inertia_:12.0f}  iters={km_plus.n_iter_:>3}  time={t_plus:.3f}s"
)
print(
    f"    Random:    inertia={km_random.inertia_:12.0f}  iters={km_random.n_iter_:>3}  time={t_random:.3f}s"
)
print("    k-means++ spreads the seed centroids apart — faster and lower inertia.")

km_labels = km_plus.predict(X_scaled)



### Checkpoint 2


In [ ]:
assert (
    km_plus.inertia_ <= km_random.inertia_ + 1
), "Task 3: k-means++ should achieve inertia at least as good as random init"
assert len(set(km_labels.tolist())) == best_k, "Task 3: wrong cluster count"
print("\n  [ok] Checkpoint 2 passed — k-means++ confirmed superior\n")



## TASK 4 — VISUALISE: Silhouette curves and per-sample silhouette


In [ ]:
# The elbow + silhouette plot is the canonical diagnostic for K.
# Per-sample silhouette reveals which points are mis-assigned — a negative
# s(i) means point i is closer to a different cluster than its own.

viz = ModelVisualizer()
history = {
    "Silhouette": sweep["silhouette"],
    "Inertia (scaled)": [i / max(sweep["inertia"]) for i in sweep["inertia"]],
}
fig = viz.training_history(history, x_label="K")
fig.update_layout(title=f"K-means: Silhouette and Inertia vs K (best K={best_k})")
fig.write_html(str(out_path("01_kmeans_elbow.html")))
print(f"  Saved: {out_path('01_kmeans_elbow.html')}")

# Per-sample silhouette
sil_samples = silhouette_samples(X_scaled, km_labels)
print(f"\n  Per-Sample Silhouette (K={best_k}):")
for cid in range(best_k):
    mask = km_labels == cid
    s = sil_samples[mask]
    n_neg = int((s < 0).sum())
    print(
        f"    Cluster {cid}: n={int(mask.sum()):>5}  mean_sil={s.mean():.4f}  "
        f"mis-assigned={n_neg} ({n_neg/len(s):.1%})"
    )
print("  Points with negative silhouette are candidates for re-review.")



### Checkpoint 3


In [ ]:
assert sil_samples.shape[0] == n_samples, "Task 4: per-sample silhouette missing points"
print("\n  [ok] Checkpoint 3 passed — visualisation and per-sample audit done\n")



## TASK 5 — APPLY: Shopee Singapore Loyalty Segmentation


In [ ]:
# SCENARIO: Shopee SG's CRM team wants to replace its hand-coded "Bronze /
# Silver / Gold / Platinum" loyalty tiers with data-driven segments. The
# existing tiers are purely revenue-based; they miss the difference
# between a "high-frequency low-basket" browser and a "low-frequency
# high-basket" infrequent whale.
#
# Why K-means is the right tool here:
#   - The customer features (recency, frequency, monetary, basket size)
#     form roughly spherical, non-overlapping clusters in z-space
#   - K is small (4-6 tiers) — K-means converges in seconds on 6K customers
#   - The centroid IS the segment profile — trivially explainable to the
#     marketing team ("Cluster 2 is 2.1σ above average on frequency")
#   - Re-segmentation runs nightly; linear cost scales to 10M customers
#
# BUSINESS IMPACT: Shopee's published ARPU for its SG marketplace is
# ~S$180/year per active buyer. A well-tuned tier system lifts campaign
# response rates by 15-25% because the offers match actual spending
# behaviour. On a 3M-buyer base, a 20% lift on S$20/buyer incremental
# campaign revenue is:
#     3,000,000 × S$20 × 0.20 = S$12M / year
# vs. effectively zero engineering cost (one silhouette sweep, one
# production job). The K-means model itself retrains in ~2 seconds.

print("  APPLY — Shopee SG Loyalty Segmentation")
print("  ─────────────────────────────────────────────────────────────────")
segment_sizes = np.bincount(km_labels)
for i, n in enumerate(segment_sizes):
    pct = n / n_samples * 100
    print(f"    Segment {i}: {n:>5,} customers ({pct:5.1f}%)")
print("    Each centroid is a 'typical customer profile' for its segment.")
print("    Marketing takes these profiles and designs tier-specific offers.")
print("    Estimated annual lift: S$12M (3M buyers × S$20 × 20%).")



### Checkpoint 4


In [ ]:
assert segment_sizes.min() > 0, "Task 5: every segment must have at least one customer"
assert (
    int(segment_sizes.sum()) == n_samples
), "Task 5: segment counts must sum to n_samples"
print("\n  [ok] Checkpoint 4 passed — segment sizes valid\n")



## REFLECTION


[x] K-means minimises within-cluster sum of squares via alternating
      assignment/update steps — guaranteed to converge
  [x] k-means++ seeding beats random init on both speed and final inertia
  [x] Silhouette score gives an objective criterion for choosing K
      (the elbow alone is subjective)
  [x] Per-sample silhouette exposes mis-assigned points for re-review
  [x] Mapped K={best_k} clusters onto a Shopee SG loyalty tier system
      with an estimated S$12M / year campaign revenue lift

  KEY INSIGHT: K-means gives you the centroids for free. The centroids
  ARE the segment profiles — no extra analysis needed before handing them
  to marketing. This is why K-means is the default first-pass clustering
  algorithm for customer segmentation.

  Next: 02_hierarchical.py — when you need a dendrogram instead of a K.


In [ ]:
print("=" * 70)
print("  WHAT YOU'VE MASTERED")
print("=" * 70)
print(
)

